# Ensemble OCR - Colab Test Notebook

Bu notebook, birden fazla OCR motorunu karsilastirmak icin kullanilir.

**Motorlar:**
- EasyOCR
- PaddleOCR
- Donut
- GOT-OCR

## 1. Kurulum

In [1]:
# GPU kontrolu
!nvidia-smi

Wed Jan 14 11:25:49 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   48C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
# Repoyu klonla
!git clone https://github.com/mervekomur/OCR-PROJECT.git
%cd OCR-PROJECT

Cloning into 'OCR-PROJECT'...
remote: Enumerating objects: 67, done.
remote: Counting objects: 100% (67/67), done.
remote: Compressing objects: 100% (48/48), done.
remote: Total 67 (delta 22), reused 62 (delta 17), pack-reused 0 (from 0)
Receiving objects: 100% (67/67), 56.24 KiB | 959.00 KiB/s, done.
Resolving deltas: 100% (22/22), done.
/content/OCR-PROJECT


In [3]:
# Temel bagimliklar
!pip install easyocr opencv-python-headless Pillow -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 47.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.7/180.7 kB 20.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 978.2/978.2 kB 70.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 300.6/300.6 kB 33.7 MB/s eta 0:00:00


In [ ]:
# PaddleOCR (opsiyonel)
!pip install paddlepaddle-gpu paddleocr -q

In [ ]:
# Donut & GOT-OCR (opsiyonel)
!pip install torch transformers -q

## 2. Gorsel Yukleme

In [ ]:
from google.colab import files
from IPython.display import Image, display

# Fis/fatura gorseli yukle
uploaded = files.upload()
image_path = list(uploaded.keys())[0]
print(f"Yuklenen dosya: {image_path}")

# Gorseli goster
display(Image(image_path, width=400))

## 3. Ensemble OCR Testi

In [ ]:
import sys
sys.path.insert(0, 'src')

from engines import EnsembleOCR, compare_engines

# Mevcut motorlari kontrol et
ensemble = EnsembleOCR()
print("Aktif motorlar:", ensemble.get_available_engines())

In [ ]:
# Tum motorlarla karsilastirma
result = compare_engines(image_path, show_table=True)

In [ ]:
# En iyi sonuc
print(f"\nEn iyi motor: {result.best_engine}")
print(f"\nCikarilan veriler:")

best_result = result.results[result.best_engine]
for key, val in best_result.fields.items():
    if not key.endswith('_confidence') and val:
        conf = best_result.fields.get(f'{key}_confidence', 0)
        print(f"  {key}: {val} ({conf:.1%})")

## 4. Tek Motor Testi

In [ ]:
# Sadece EasyOCR
from engines import EasyOCREngine

engine = EasyOCREngine()
result = engine.extract(image_path)

print(f"Merchant: {result.fields.get('merchant')}")
print(f"Date: {result.fields.get('date')}")
print(f"Total: {result.fields.get('total')}")
print(f"Confidence: {result.confidence:.1%}")

In [ ]:
# Raw OCR metni
print("=" * 50)
print("RAW OCR TEXT:")
print("=" * 50)
print(result.raw_text)

## 5. Sonuclari JSON Olarak Kaydet

In [ ]:
import json

# Karsilastirma sonuclarini JSON'a cevir
ensemble = EnsembleOCR()
comparison = ensemble.process(image_path)
json_result = ensemble.to_json(comparison)

# Dosyaya kaydet
with open('ocr_result.json', 'w', encoding='utf-8') as f:
    json.dump(json_result, f, ensure_ascii=False, indent=2)

print("Sonuclar ocr_result.json dosyasina kaydedildi.")

# Indir
files.download('ocr_result.json')